# Sandbox Agent + Sandbox Playground

Use this notebook to test both paths from a local machine:

1. Agent path (through backend `/api/chat`, with mode switch between `local` and `cluster`)
2. Direct sandbox path (using `k8s_agent_sandbox.SandboxClient` against the sandbox router)


## Prerequisites

- App running locally: `./apps/sandboxed-react-agent/dev-sandbox.sh up local` or `up cluster`
- Optional router port-forward for cluster mode / direct sandbox:
  `./apps/sandboxed-react-agent/dev-sandbox.sh port-forward start`
- Python deps in notebook environment:
  `pip install requests k8s-agent-sandbox`


In [ ]:
import json
import os
import socket
import textwrap
from urllib.parse import urlparse
from typing import Any

import requests

BASE_API = "http://localhost:8080/api"
BACKEND_ROUTER_URL_FOR_COMPOSE = os.getenv("BACKEND_ROUTER_URL_FOR_COMPOSE", "http://host.docker.internal:18080")
DIRECT_ROUTER_URL = os.getenv("DIRECT_ROUTER_URL", "http://127.0.0.1:18080")
DEFAULT_TEMPLATE = "python-runtime-template-small"
DEFAULT_NAMESPACE = "alt-default"
DEFAULT_SERVER_PORT = 8888


In [ ]:
def health() -> dict[str, Any]:
    response = requests.get(f"{BASE_API}/health", timeout=15)
    response.raise_for_status()
    return response.json()


def get_config() -> dict[str, Any]:
    response = requests.get(f"{BASE_API}/config", timeout=30)
    response.raise_for_status()
    return response.json()


def set_agent_mode(mode: str, sandbox_api_url: str | None = None) -> dict[str, Any]:
    payload: dict[str, Any] = {"sandbox_mode": mode}
    if sandbox_api_url:
        payload["sandbox_api_url"] = sandbox_api_url
    response = requests.post(f"{BASE_API}/config", json=payload, timeout=30)
    response.raise_for_status()
    return response.json()


def chat(message: str, session_id: str | None = None) -> dict[str, Any]:
    payload = {"message": message, "session_id": session_id}
    response = requests.post(f"{BASE_API}/chat", json=payload, timeout=180)
    response.raise_for_status()
    return response.json()


def is_router_reachable(api_url: str, timeout_seconds: float = 1.0) -> bool:
    parsed = urlparse(api_url)
    host = parsed.hostname
    port = parsed.port
    if not host or not port:
        return False
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.settimeout(timeout_seconds)
        return sock.connect_ex((host, port)) == 0


def assert_router_reachable(api_url: str) -> None:
    if is_router_reachable(api_url):
        return
    raise RuntimeError(
        f"Router {api_url} is not reachable from this notebook kernel. \
Start/restart port-forward with ./apps/sandboxed-react-agent/dev-sandbox.sh port-forward restart. \
If your notebook runs in a container, use DIRECT_ROUTER_URL=http://host.docker.internal:18080."
    )


In [ ]:
print(health())
print(json.dumps(get_config(), indent=2))
print(f"Direct router URL: {DIRECT_ROUTER_URL}")
print(f"Router reachable now: {is_router_reachable(DIRECT_ROUTER_URL)}")

## Agent path: local mode

Runs tools inside the backend container/process.

In [ ]:
set_agent_mode("local")

In [ ]:
local_result = chat("Use sandbox_exec_python to print Python version and working directory.")
print(json.dumps(local_result, indent=2))

## Agent path: cluster mode

Before running this section, start router port-forward in a terminal:

`./apps/sandboxed-react-agent/dev-sandbox.sh port-forward start`

For docker-compose backend, router URL should be `http://host.docker.internal:18080`.


In [ ]:
set_agent_mode("cluster", BACKEND_ROUTER_URL_FOR_COMPOSE)

In [ ]:
cluster_result = chat("Use sandbox_exec_shell to run: uname -a && python --version")
print(json.dumps(cluster_result, indent=2))

## Direct sandbox path (bypass agent)

This uses `SandboxClient` directly against the router, so you can see raw command behavior.

If this notebook kernel runs inside a container, set `DIRECT_ROUTER_URL` to `http://host.docker.internal:18080` before starting Jupyter.

In [ ]:
from k8s_agent_sandbox import SandboxClient


def direct_sandbox_run(
    command: str,
    api_url: str = DIRECT_ROUTER_URL,
    template_name: str = DEFAULT_TEMPLATE,
    namespace: str = DEFAULT_NAMESPACE,
    server_port: int = DEFAULT_SERVER_PORT,
    sandbox_ready_timeout: int = 420,
    gateway_ready_timeout: int = 180,
):
    assert_router_reachable(api_url)
    with SandboxClient(
        template_name=template_name,
        api_url=api_url,
        namespace=namespace,
        server_port=server_port,
        sandbox_ready_timeout=sandbox_ready_timeout,
        gateway_ready_timeout=gateway_ready_timeout,
    ) as sandbox:
        result = sandbox.run(command)
        return {
            "stdout": getattr(result, "stdout", ""),
            "stderr": getattr(result, "stderr", ""),
            "returncode": getattr(result, "returncode", None),
        }

In [ ]:
assert_router_reachable(DIRECT_ROUTER_URL)
print(f"Router reachable from notebook kernel at {DIRECT_ROUTER_URL}")

In [ ]:
direct_shell = direct_sandbox_run("python --version && uname -a")
print(json.dumps(direct_shell, indent=2))

In [ ]:
python_command = textwrap.dedent(
    """
    python - <<'PY'
    import math
    values = [round(math.sin(i / 10), 4) for i in range(10)]
    print(values)
    PY
    """
).strip()
direct_python = direct_sandbox_run(python_command)
print(json.dumps(direct_python, indent=2))

## Notes

- Agent `local` mode is useful for fast iteration but does not provide sandbox isolation.
- Agent `cluster` mode and direct `SandboxClient` both use the remote router and runtime template.
- You can switch agent mode at runtime using `set_agent_mode(...)` without restarting compose.
